# Evaluate the saved GRPO adapter on a Colab T4
Replays the recorded 32-question paired evaluation (base model with the adapter disabled versus enabled) using the adapter bundled in `adapter/`. No training is performed. The recorded replay already in `results/replay/` reproduced all 64 original response texts on a Tesla T4.

1. Open a T4 GPU session, run the installation cell, then select **Runtime → Restart session**.
2. Skip installation and run the remaining cells in order.
3. The project is fetched with `git clone` when `REPO_URL` is set. Otherwise you are prompted to upload the repository ZIP (GitHub's **Code → Download ZIP** works).

Leave `ADAPTER_DIR` unchanged; the hash check rejects any other adapter. Outputs are written to a new Drive folder and never overwrite the recorded results. The evaluator has a 20-minute soft budget: if it prints `PAUSED`, rerun the last cell to continue from the saved answers.


In [ ]:
%pip install "transformers==4.56.2" "peft==0.17.1" "accelerate==1.10.1" "datasets==4.1.1" "bitsandbytes==0.48.1"
# Restart session now, then skip this cell.

In [ ]:
from google.colab import files,drive
drive.mount('/content/drive')
from pathlib import Path
import zipfile,sys,subprocess,json,stat
REPO_URL=''  # Optional: this repository's GitHub URL. Leave empty to upload a ZIP.
if REPO_URL:
    ROOT=Path('/content/Chasing-the-Aha-Moment')
    if not ROOT.exists():
        subprocess.run(['git','clone','--depth','1',REPO_URL,str(ROOT)],check=True)
    assert (ROOT/'scripts/evaluate.py').exists(),'Expected the project root'
else:
    uploaded=files.upload()  # Select the repository ZIP
    assert len(uploaded)==1,'Upload only the project ZIP'
    archive=Path(next(iter(uploaded)))
    assert zipfile.is_zipfile(archive),'Expected a ZIP file'
    DEST=Path('/content/aha_project').resolve();DEST.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        assert sum(i.file_size for i in z.infolist())<50*1024*1024,'Unexpectedly large project archive'
        for info in z.infolist():
            target=(DEST/info.filename).resolve()
            assert target==DEST or DEST in target.parents,'Unsafe archive path'
            assert not stat.S_ISLNK(info.external_attr>>16),'Symlink archive entry rejected'
        z.extractall(DEST)
    # GitHub source ZIPs include the branch name in their top-level directory.
    with zipfile.ZipFile(archive) as z:
        roots=[Path(i.filename).parent.parent for i in z.infolist()
               if Path(i.filename).parts[-2:]==('scripts','evaluate.py')]
    assert len(roots)==1,'Expected one project root containing scripts/evaluate.py'
    ROOT=DEST/roots[0]
    assert (ROOT/'scripts/evaluate.py').exists(),'Expected the project root'
subprocess.run([sys.executable,str(ROOT/'scripts/verify_results.py')],check=True)
print('Project verified:',ROOT)

In [ ]:
ADAPTER_DIR = str(ROOT/'adapter')  # Bundled with the repository.
sys.path.insert(0,str(ROOT/'src'))
from artifact_checks import verify_adapter
protocol=json.loads((ROOT/'results/evaluation/protocol.json').read_text())
print(verify_adapter(ADAPTER_DIR,protocol))
OUTPUT=Path('/content/drive/MyDrive/aha_adapter_replay')
print('Replay outputs:',OUTPUT)


In [ ]:
# Re-run this cell after PAUSED; it skips saved answers.
subprocess.run([sys.executable,str(ROOT/'scripts/evaluate.py'),
    '--adapter',ADAPTER_DIR,'--output',str(OUTPUT),'--budget-seconds','1200'],check=True)
